# 🔴 Solution: Top-p Sampling

### Exercise contract
Accept logits of shape `(vocab_size,)` or `(B, vocab_size)` and return an integer token-index tensor of shape `()` or `(B,)`, respectively. Sample each row independently. Any algorithm sampling the specified categorical distribution is valid; the judge checks approximate frequencies and allowed tokens, **not identical draws for a fixed seed**.

Assume `0 < p <= 1`. Keep the smallest descending-probability prefix whose cumulative mass is **at least** p: stop at equality, not one token later. If probabilities tie, any consistent ordering is valid. At p=1, sample from the full distribution.

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def top_p_sample(logits: torch.Tensor, p: float) -> torch.Tensor:
    squeeze = logits.dim() == 1
    if squeeze:
        logits = logits.unsqueeze(0)

    probs = torch.softmax(logits, dim=-1)
    outputs = []
    for row in probs:
        sorted_probs, sorted_indices = torch.sort(row, descending=True)
        cumulative = sorted_probs.cumsum(dim=0)
        # Keep a token only if the preceding prefix has not yet reached p.
        keep = torch.cat((torch.ones(1, dtype=torch.bool, device=row.device),
                          cumulative[:-1] < p))

        filtered = torch.zeros_like(row)
        filtered[sorted_indices[keep]] = row[sorted_indices[keep]]
        filtered = filtered / filtered.sum()
        outputs.append(torch.multinomial(filtered, num_samples=1).squeeze(0))

    outputs = torch.stack(outputs)
    return outputs.squeeze(0) if squeeze else outputs


In [ ]:
# Verify
torch.manual_seed(0)
logits = torch.tensor([[4.0, 3.5, 1.0, -2.0]])
print(top_p_sample(logits, p=0.8))
